## Spatial transcriptomics platforms

Spatial transcriptomics platforms fall into two families, and most of the practical differences follow from which family a platform belongs to.

### 1. Sequencing-based (spatial barcode capture)

These platforms put a grid or bead array of position-barcoded oligos under the tissue. Transcripts, or probes hybridized to them, are captured and tagged with their location, then read out on an ordinary NGS sequencer. The result is unbiased, whole-transcriptome data, but each spot captures only a small fraction of the molecules present, and the spots are defined by the array rather than by cells.

- **Visium (v1, and v2/CytAssist).** 
  - Spots are 55 µm wide, 100 µm apart center to center, with gaps between them. 
  - Each spot holds about 1–10 cells, so every measurement is a mixture. 
  - This is the setting where deconvolution (cell2location, RCTD, Tangram, CARD, etc.) is required, and it is conceptually close to bulk deconvolution. 
  - Fresh-frozen tissue uses poly-dT capture;  Formalin-Fixed Paraffin-Embedded (FFPE) uses probe-based capture.

- **Visium HD.** It uses a gapless lawn of 2 µm squares and is whole-transcriptome (probe-based), over a 6.5 × 6.5 mm capture area. In practice you bin to 8 µm or segment cells from the H&E image. A 3′ (poly-dT) version also exists for fresh-frozen tissue.
- **Slide-seq / Slide-seqV2 (Broad; sold commercially as Curio Seeker).** About 10 µm barcoded beads are packed randomly on a puck, and each bead's position is decoded by in situ sequencing before the experiment. Resolution is near single cell, but sensitivity per bead is low (V2 improved it considerably), the capture area is small (about 3 mm diameter), and you can't do a tidy same-section H&E. RCTD was originally developed for Slide-seq data.

- **Stereo-seq (BGI/STOmics).** It uses DNA nanoball arrays with very high feature density. Its feature resolution is 220 nm, close to the subcellular precision of imaging platforms, but because it is still array-based, transcripts are assigned by spatial binning rather than by cell boundaries, so mixed-cell signal can persist. Its big advantage is very large capture areas (up to centimeter scale).
- **Others in this family:** HDST, Seq-Scope, and DBiT-seq. DBiT-seq is microfluidic and well suited to spatial multi-omics, such as ATAC and protein.
- **GeoMx DSP** is a different design. You pick regions of interest (ROIs) with morphology markers and profile each ROI as a whole. It is not a grid, and its resolution is at the ROI level.

### 2. Imaging-based (in situ hybridization or sequencing)

Here a predefined gene panel is detected by repeated rounds of fluorescent hybridization and optical barcode decoding. Platforms in this family (Xenium, MERSCOPE, CosMx) detect individual transcripts at subcellular positions in intact tissue. You get the true x,y coordinate of each molecule, high sensitivity per gene, and real single-cell data once cells are segmented. The cost is that you only see the genes in your panel.

- **Xenium (10x).** Panels run about 480 genes (tissue panels) or 5,000 genes (5K panel), with an imaging area of about 10.45 × 22.45 mm. Chemistry uses padlock probes with rolling-circle amplification.
- **CosMx SMI (now Bruker, formerly NanoString).** It offers 1K and 6K panels and a whole-transcriptome panel. It is strong on protein co-detection.
- **MERSCOPE (Vizgen).** Based on MERFISH, which uses error-robust binary barcodes. It is excellent for very large fields of view on fresh-frozen tissue.
- **Academic predecessors:** seqFISH+, MERFISH, STARmap, and ExSeq.

### Head-to-head summary

| | Visium | Visium HD | Slide-seqV2 | Stereo-seq | Xenium / CosMx / MERSCOPE |
|---|---|---|---|---|---|
| Readout | NGS | NGS | NGS | NGS | Optical imaging |
| Unit | 55 µm spot | 2 µm square (binned) | ~10 µm bead | 0.22–0.5 µm features (binned) | Single molecule |
| Genes | Whole transcriptome | Whole transcriptome | Whole transcriptome | Whole transcriptome | Panel (≈300–6K; CosMx up to whole transcriptome) |
| Single cell? | No (needs deconvolution) | ~Yes (binning or segmentation) | ~Near | ~Yes (binning) | Yes (segmentation) |
| Main weakness | Mixed spots | Sparsity, bin/segmentation choice | Low sensitivity, small area | Sparsity, lateral diffusion | Panel bias, segmentation errors, cost and time |

### Practical points for analysis

- **Sensitivity versus breadth is the core trade-off.** Sequencing platforms find unexpected genes but miss many molecules per cell. Imaging platforms detect panel genes well but can't discover anything outside the panel. A common design is to use Visium or Visium HD for discovery and Xenium or CosMx for validation at cellular resolution, with matched scRNA-seq as the reference.
- **Benchmarks on the same tissue.** A 2026 study profiled matched FFPE sections from six cancer types on Visium v1, Visium v2/CytAssist, Visium HD, Xenium and CosMx. It found that Xenium had consistently lower background noise and stronger spatial signal than CosMx. When the authors combined Visium protein data with matched RNA, they found widespread RNA–protein decoupling. A second benchmark, by Luo, Zeng et al. in Nature Communications (2025), compared Stereo-seq v1.3, Visium HD FFPE, CosMx 6K and Xenium 5K on colon, liver and ovarian tumors, using adjacent-section CODEX and scRNA-seq as ground truth. Its SPATCH dataset is a good resource for testing methods.
- **The main computational problem shifts with resolution.** At Visium resolution, the hard part is deconvolution. At HD or Stereo-seq resolution, it becomes binning and segmentation plus correction for lateral diffusion (methods like bin2cell, ENACT). For imaging platforms, it is segmentation (Baysor, Cellpose, proseg) and panel-aware annotation.
- **Protein co-detection:** Visium and Visium HD can measure 35 proteins with the CytAssist Gene and Protein assay, Stereo-CITE extends Stereo-seq to 100+ proteins, and GeoMx profiles 570+ proteins per ROI.

For tumor microenvironment questions in FFPE tissue such as PDAC, a reasonable 2026 default is Visium HD for unbiased mapping of stroma and immune niches, with Xenium 5K or CosMx 6K for single-cell neighborhood and ligand–receptor analysis.

Sources:
- [Benchmark of spatial platforms across six cancer types (Genome Biology, 2026)](https://fis.tu-dresden.de/portal/en/publications/a-technical-comparison-of-spatial-transcriptomics-platforms-across-six-cancer-types(51cca1f8-4d19-4859-be00-35979b182bef).html)
- [Benchmarking subcellular spatial platforms across human tumors (Luo, Zeng et al.)](https://researchexchange.ocrahope.org/papers/s41467-025-64292-3)
- [Review of spatial transcriptomics platforms in ovarian biology (2026)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC13403513/)
- [Kennedy Institute spatial transcriptomics facility comparison table](https://www.kennedy.ox.ac.uk/platforms-and-technologies/single-cell-and-spatial-genomics-facility/spatial-transcriptomics)
- [IntuitionLabs: Spatial biology platforms compared](https://intuitionlabs.ai/articles/spatial-biology-platforms-compared)

### single cell spatial transcriptomics database

Single-cell spatial transcriptomics databases store, harmonize, and visualize gene expression data mapped directly to tissue coordinates at single-cell or subcellular resolution. Key public repositories include STOmics DB, SpatialDB, DeepSpaceDB, and the Single Cell Portal. [1](https://db.cngb.org/stomics/), [2](https://www.biorxiv.org/content/10.64898/2026.01.15.699623v1), [3](https://pubmed.ncbi.nlm.nih.gov/37953328/), [4](https://www.completegenomics.com/learn/applications/spatial-transcriptomics/), [5](http://www.spatialomics.org/SpatialDB/), [6](https://singlecell.broadinstitute.org/)

#### Major Spatial Transcriptomics Databases

- STOmics DB (STOmics DataBase): A large-scale hub archiving multi-omics and spatiotemporal atlases across species like human and mouse, featuring cell type annotation and visualization tools. [1](https://pubmed.ncbi.nlm.nih.gov/37953328/), [2](https://db.cngb.org/stomics/)

- DeepSpaceDB (DeepSpaceDB 2.0): An interactive platform specialized in high-throughput imaging data from platforms like 10x Xenium, offering browser-based gene expression and region-of-interest exploration. [1](https://www.biorxiv.org/content/10.64898/2026.01.15.699623v1)
- SpatialDB (SpatialDB): An early public database dedicated to curating published spatial gene expression profiles, side-by-side dataset comparisons, and spatially variable gene analyses. [1](http://www.spatialomics.org/SpatialDB/)
- Single Cell Portal (Single Cell Portal): Developed by the Broad Institute, this portal hosts extensive single-cell and spatial datasets (including MERFISH and Xenium) with downloadable count matrices and interactive plots. [1](https://singlecell.broadinstitute.org/)


### Slide-seq 

Slide-seq is a high-resolution spatial transcriptomics technology developed by researchers at the Broad Institute of MIT and Harvard (Fei Chen and Evan Macosko). It measures genome-wide gene expression within tissue sections while retaining the precise spatial location of cells. [1](https://www.science.org/doi/10.1126/science.aaw1219), [2](https://www.broadinstitute.org/news/new-tool-uses-rna-sequencing-chart-rich-maps-cellular-and-tissue-function), [3](https://commonfund.nih.gov/earlyindependence/highlights/slide-seq-novel-method-map-tissue-cells), [4](https://www.broadinstitute.org/publications/broad508511)

How It Works

* DNA-Barcoded Beads ("Pucks"): A dense monolayer of 10-micron DNA-barcoded beads (comparable to the size of individual cells) is packed onto a surface called a puck. [1](https://pubmed.ncbi.nlm.nih.gov/30923225/), [2](https://www.nature.com/articles/s41587-020-0739-1), [3](https://www.science.org/doi/10.1126/science.aaw1219)
* RNA Transfer: Fresh-frozen tissue sections are placed directly onto the puck, allowing cellular RNA to transfer onto the underlying beads. [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC11254742/), [2](https://pubmed.ncbi.nlm.nih.gov/30923225/)
* Sequencing & Mapping: Standard high-throughput RNA sequencing is performed. By matching the RNA sequences back to the known spatial coordinates of the bead barcodes, researchers map gene expression across the tissue without needing a microscope for primary localization. [1](https://www.youtube.com/watch?v=mQXr7ZsojpE&t=213), [2](https://www.broadinstitute.org/news/new-tool-uses-rna-sequencing-chart-rich-maps-cellular-and-tissue-function)

Key Features & Versions

* Original Slide-seq (2019): Achieved 10-micron spatial resolution and enabled 3D tissue volume reconstruction (such as mapping the mouse hippocampus and cerebellum). [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC8606189/), [2](https://www.science.org/doi/10.1126/science.aaw1219), [3](https://www.broadinstitute.org/publications/broad508511)
* Slide-seqV2 (2020): Improved bead synthesis, library generation, and array indexing to capture roughly 50% of single-cell RNA-seq efficiency—a ~10-fold increase in transcript capture over the original method. [1](https://www.nature.com/articles/s41587-020-0739-1), [2](https://www.youtube.com/watch?v=mQXr7ZsojpE&t=213)
* Computational Pipeline: Processed using open-source, cloud-optimized workflows like the Broad Institute WARP Slide-seq Pipeline available on platforms like Terra. Commercialized assays are also known under tools like Curio Seeker. [1](https://broadinstitute.github.io/warp/docs/Pipelines/SlideSeq_Pipeline/README)



### ScanPy

### SquidPy

https://squidpy.readthedocs.io/en/stable/index.html

### Cell2Location

Cell2location package is implemented in a general way (using https://pyro.ai/ and https://scvi-tools.org/) to support multiple related models - both for spatial mapping and estimating reference cell type signatures:

1. Cell2location for **spatial mapping of cell types** which **estimates cell abundance by decomposing spatial data** into **reference expression signatures of cell types** (LocationModelLinearDependentWMultiExperimentLocationBackgroundNormLevelGeneAlphaPyroModel).

2. Models for estimating reference expression signatures of cell types from scRNA data, accounting for variable sequencing depth between batches (e.g. 10X reaction), additive background (contaminating RNA), multiplicative platform effect between scRNA technologies.

3. Cell2location model for mapping to Nanostring WTA data (LocationModelWTA). See https://github.com/vitkl/SpaceJam for a new more versatile version.

4. Similified versions of model #1 that lack particular features of the full model, accessible from cell2location.models.simplified

Additionally we provide 2 models for downstream analysis of cell abundance estimates, accessible from cell2location.models.downstream:

1. CoLocatedGroupsSklearnNMF - identifying groups of cell types with similar locations using NMF (wrapper around sklearn NMF). See tutorial for usage.

2. ArchetypalAnalysis - identifying smoothly varying and mutually exclusive tissue zones with Archetypa Analysis.

https://cell2location.readthedocs.io/en/latest/index.html

https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_tutorial.html#

### Cell-Cell Communication (CCC)

The diversification of methods for studying cell-cell interactions and communication  
https://pubmed.ncbi.nlm.nih.gov/38238518/

Armingol E, Baghdassarian HM, Lewis NE. The diversification of methods for studying cell-cell interactions and communication. Nat Rev Genet. 2024 Jun;25(6):381-400. doi: 10.1038/s41576-023-00685-8. Epub 2024 Jan 18. PMID: 38238518; PMCID: PMC11139546.


### Liana

https://liana.readthedocs.io/en/stable/

https://saezlab.github.io/liana/

https://github.com/scverse/liana

Paper: Dimitrov D, Schäfer PSL, Farr E, Rodriguez-Mier P, Lobentanzer S, Badia-I-Mompel P, Dugourd A, Tanevski J, Ramirez Flores RO, Saez-Rodriguez J. LIANA+ provides an all-in-one framework for cell-cell communication inference. Nat Cell Biol. 2024 Sep;26(9):1613-1622. doi: 10.1038/s41556-024-01469-w. Epub 2024 Sep 2. PMID: 39223377; PMCID: PMC11392821.

https://pubmed.ncbi.nlm.nih.gov/39223377/


#### LIANA+

LIANA+ (Ligand-Acetate Network Analysis / All-in-one cell-cell communication framework) is an all-in-one computational framework published in Nature Cell Biology that infers and decodes cell–cell communication (CCC) from single-cell, spatial, and multi-omics data. [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC11392821/), [2](https://pubmed.ncbi.nlm.nih.gov/39223377/)

Key Details & Publications

- Core Framework (LIANA+): Published in Nature Cell Biology (September 2024), providing scalable inference for inter- and intracellular signaling.

- Integrated Knowledge Bases: Combines resources such as OmniPath, BioCypher, and MetalinksDB for protein- and metabolite-mediated interactions.
- Spatial & Multi-Condition Analysis: Supports spatial metrics (e.g., spatially weighted cosine similarity), multi-view modeling, PyDESeq2 differential analysis, and downstream tensor decomposition (Tensor-cell2cell integration on PubMed).
- Software Implementation: Available via GitHub at the liana-py Repository with documentation at liana-py ReadTheDocs. [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC10168343/), [2](https://pubmed.ncbi.nlm.nih.gov/38631346/), [3](https://pubmed.ncbi.nlm.nih.gov/39223377/), [4](https://pmc.ncbi.nlm.nih.gov/articles/PMC11392821/)
If you are looking for specific code implementation, a particular method within LIANA+, or a comparison with other tools like NicheNet, let me know!





### CellPhoneDB

https://cellphonedb.readthedocs.io/en/latest/index.html

Troulé K, Petryszak R, Cakir B, Cranley J, Harasty A, Prete M, Tuong ZK, Teichmann SA, Garcia-Alonso L, Vento-Tormo R. CellPhoneDB v5: inferring cell-cell communication from single-cell multiomics data. Nat Protoc. 2025 Dec;20(12):3412-3440. doi: 10.1038/s41596-024-01137-1. Epub 2025 Mar 25. PMID: 40133495.


### Single Sample Gene Set Enrichment Analysis (ssGSEA

Single-sample Gene Set Enrichment Analysis (ssGSEA) is an variation of the GSEA algorithm that instead of calculating enrichment scores for groups of samples (i.e Control vs Disease) and sets of genes (i.e pathways), it provides a score for each each sample and gene set pair (https://www.genepattern.org/modules/docs/ssGSEAProjection/4).

In CKG this function is implemented in the function: run_ssgsea (analytics_core.analytics).

https://ckg.readthedocs.io/en/latest/notebooks/recipes/ssGSEA_with_PCA.html

https://github.com/broadinstitute/ssGSEA2.0


#### GSEA-msigdb

https://www.gsea-msigdb.org/gsea/msigdb




### In Situ Sequencing (ISS)

In Situ Sequencing (ISS) is a spatial transcriptomics technique that sequences mRNA directly inside fixed tissue or cell samples while preserving their native cellular and subcellular locations. [1](https://www.biocompare.com/Editorial-Articles/565636-Introduction-to-In-Situ-Sequencing/), [2](https://omicsveu.com/our-blog/in-situ-sequencing-benefits-applications-and-protocol/)

Core Workflow & Methodology

- Reverse Transcription: RNA is converted into complementary DNA (cDNA) right inside the intact tissue. [1](https://frontlinegenomics.com/on-the-horizon-in-situ-sequencing/), [2](https://www.youtube.com/watch?v=FTDBxvt6yI0&vl=en&t=153)
- Padlock Probes: Specific, target-matching DNA probes hybridize to the cDNA sequences and circularize. [1](https://www.biocompare.com/Editorial-Articles/565636-Introduction-to-In-Situ-Sequencing/), [2](https://frontlinegenomics.com/on-the-horizon-in-situ-sequencing/)
- Rolling Circle Amplification (RCA): Enzymes amplify the circularized probes into dense DNA clusters called Rolling Circle Products (RCPs). [1](https://www.biocompare.com/Editorial-Articles/565636-Introduction-to-In-Situ-Sequencing/), [2](https://frontlinegenomics.com/on-the-horizon-in-situ-sequencing/)
- Decoding / Sequencing: RCPs are read out in situ using methods like sequencing-by-ligation, sequencing-by-hybridization (e.g., HybISS), or sequencing-by-synthesis, followed by wide-field or confocal microscopy. [1](https://www.scilifelab.se/units/in-situ-sequencing/), [2](https://www.biocompare.com/Editorial-Articles/627052-Choosing-an-In-Situ-Sequencing-Method/), [3](https://www.biocompare.com/Editorial-Articles/565636-Introduction-to-In-Situ-Sequencing/)

Key Advantages

- Spatial Resolution: Maps hundreds to thousands of gene transcripts at single-cell and subcellular resolutions.
- Tissue Compatibility: Works with both fresh-frozen and formalin-fixed, paraffin-embedded (FFPE) samples.
- Histological Context: Keeps cells in their structural environment, avoiding the loss of spatial data that happens with bulk or dissociated single-cell RNA sequencing. [1](https://www.nature.com/articles/s42003-025-08518-6), [2](https://omicsveu.com/our-blog/in-situ-sequencing-benefits-applications-and-protocol/), [3](https://www.scilifelab.se/units/in-situ-sequencing/), [4](https://www.biocompare.com/Editorial-Articles/565636-Introduction-to-In-Situ-Sequencing/)

Main Applications

- Cell Type Mapping: Identifying and classifying cell types within complex tissue atlases (e.g., brain, tumor microenvironments).
- Disease Pathology: Studying localized disease mechanisms, such as neurodegenerative plaques in Alzheimer's disease or tumor heterogeneity.
- Drug Target Validation: Pinpointing where specific therapeutic targets are expressed in tissues. [1](https://www.science.org/content/webinar/every-cell-has-its-place-situ-sequencing-tissue-samples-single-cell-resolution), [2](https://www.biocompare.com/Editorial-Articles/627052-Choosing-an-In-Situ-Sequencing-Method/), [3](https://omicsveu.com/our-blog/in-situ-sequencing-benefits-applications-and-protocol/)
If you are planning an experiment or data analysis, let me know:
- Are you working with fresh-frozen or FFPE tissue?
- Are you looking for a targeted gene panel or whole-transcriptome approach?


### iStar, TESLA, and XFuse

iStar, TESLA, and XFuse are all prominent, deep-learning-based algorithms used in bioinformatics for super-resolution spatial transcriptomics (ST). [1](https://www.cell.com/cell-reports-methods/fulltext/S2667-2375(24)00327-8), [2](https://www.researchgate.net/figure/)

Differential-gene-expression-and-cell-type-composition-a-b-Differential-gene-expression_fig8_356621506)
In modern genomics, standard barcode-based spatial transcriptomics (like 10x Genomics Visium) captures gene expression data at discrete spots but cannot achieve true single-cell resolution on its own. To overcome this barrier, these three computational methods integrate low-resolution spot-based gene expression data with highly detailed, high-resolution histology images (such as H&E staining) to predict and map gene expression at a much finer, "super-resolution" or pixel level. [1](https://www.researchgate.net/figure/Differential-gene-expression-and-cell-type-composition-a-b-Differential-gene-expression_fig8_356621506), [2](https://www.cell.com/cell-reports-methods/fulltext/S2667-2375(24)00327-8)

Here is how each method approaches the problem:
- XFuse (Deep Spatial Data Fusion): One of the early frameworks in this space, XFuse utilizes a deep generative model (specifically a dual-objective variational auto-encoder) to fuse spot-level ST data with histological images, inferring super-resolution gene expression profiles. [1](https://www.sciencedirect.com/science/article/pii/S2667237524003278), [2](https://academic.oup.com/bib/article-pdf/doi/10.1093/bib/bbae630/60930108/bbae630.pdf), [3](https://pmc.ncbi.nlm.nih.gov/articles/PMC13201544/)

- TESLA (Tumor Edge Structure and Lymphocyte Multi-level Annotation): This method builds high-resolution profiles by determining local neighborhood relationships directly from H&E images. It uses a Euclidean distance metric and graph-based smoothing to account for similarities in physical locations and image features between measured spots and interpolated "super-pixels". [1](https://academic.oup.com/bib/article-pdf/doi/10.1093/bib/bbae630/60930108/bbae630.pdf), [2](https://www.sciencedirect.com/science/article/pii/S2667237524003278), [3](https://www.cell.com/cell-reports-methods/fulltext/S2667-2375(24)00327-8), [4](https://pmc.ncbi.nlm.nih.gov/articles/PMC13201544/)

- iStar: A more advanced architecture that addresses limitations in feature extraction by using a hierarchical vision transformer (HViT). It extracts tissue morphology features at multiple scales from the histology images, capturing both local structures and global spatial relationships to predict super-pixel-level gene expression. [1](https://www.researchgate.net/figure/Prediction-of-spatial-gene-expression-from-histology-images-squamous-cell-carcinoma_fig4_356621506), [2](https://www.sciencedirect.com/science/article/pii/S2667237524003278), [3](https://pmc.ncbi.nlm.nih.gov/articles/PMC13201544/)
Would you like to explore a comparative performance analysis of these algorithms, or are you looking for implementation details and code repositories for a specific dataset?



### Visium


#### Visium-data-preprocessing

https://lmweber.org/Visium-data-preprocessing/index.html



## Extra

These labels come from the **Tissue** column of the scRNA reference (**lymphoid_organ_sc.h5ad**) used for cell2location.

### Lymphoid tissue
Lymphoid tissues are where lymphocytes develop and where immune responses are started.

- **Primary lymphoid organs** produce and educate B and T cells.

  - bone marrow

  - thymus

- **Secondary lymphoid organs** are where naive lymphocytes meet antigen and adaptive responses begin.

  - lymph nodes

  - spleen
  - tonsils
  - gut Peyer's patches

The reference combines secondary lymphoid organs only. 

  - It covers tonsil, spleen and three kinds of lymph node, 

  - so the 34 cell types include germinal-centre states (B_GC_DZ/LZ, TfH_GC, FDC) that adult lymph nodes alone capture poorly.

<br>

**A lymph node has three zones**, which line up with the regions in your analysis:

- **Cortex:** B-cell follicles and germinal centres (FDC, GC B cells, TfH; CXCL13→CXCR5).

- **Paracortex:** the T-cell zone (naive T cells, CCR7+ DCs, CCL19/CCL21 from fibroblastic reticular cells, high endothelial venules).
- **Medulla:** medullary cords and sinuses (plasma cells, macrophages; ACKR1+ venules).

<br>

### The three lymph-node labels

A lymph node's character depends largely on the tissue it drains, which sets the antigens it sees.

| Label | Lymph node | Drains | Typical features |
|---|---|---|---|
| **iLN** | inguinal (groin) | skin and soft tissue of the lower limb, lower abdominal wall, external genitalia | skin-type antigen exposure; often fewer active GCs in adults |
| **tLN** | thoracic (mediastinal / hilar) | lungs and airways | airborne and inhaled antigens; often contains anthracotic (pigment-laden) macrophages |
| **mLN** | mesenteric | small and large intestine | continuous exposure to microbiota and food antigens; more IgA-switched B and plasma cells, gut-homing (CCR9, α4β7) and tolerance-related T cells |

I'm confident about iLN = inguinal and mLN = mesenteric. For tLN, "thoracic" is the usual meaning in this atlas, but check the original study's metadata if the exact site matters.

### What this means for your data

- **Tissue is confounded with donor and study.** In the reference, spleen, iLN and tLN are adult only and all come from one donor (A16). Only mLN spans several ages and donors. Tissue differences are therefore not separable from donor or study effects.

- **The Visium slide has no tissue label.** 

  - **V1_Human_Lymph_Node** is a single 10x section with no site annotation. 
  - cell2location maps the pooled reference cell types onto it, 
  - so the slide's cell abundances and CCC results don't say whether it resembles an iLN, tLN or mLN.


### CD52: glycoprotein heavily expressed on mature immune cells found within secondary lymphoid organs

CD52 is a cell-surface glycoprotein heavily expressed on mature immune cells found within secondary lymphoid organs (such as lymph nodes, the spleen, and tonsils). [1](https://pubmed.ncbi.nlm.nih.gov/28283679/), [2](https://www.sciencedirect.com/topics/neuroscience/cd52), [3](https://ashpublications.org/blood/article/109/11/4856/23088/T-cell-dependent-survival-of-CD20-and-CD20-plasma)

Key Roles of CD52 in Secondary Lymphoid Tissues

- High Expression: CD52 is densely expressed on mature T lymphocytes, B lymphocytes, natural killer (NK) cells, monocytes, macrophages, and dendritic cells residing in secondary lymphoid structures. [1](https://www.sciencedirect.com/topics/pharmacology-toxicology-and-pharmaceutical-science/cd52-antigen), [2](https://pubmed.ncbi.nlm.nih.gov/28283679/)

- Survival & Cell Contact: In tissues like tonsils and lymph nodes, plasma cell and lymphocyte survival can be heavily influenced by contact-dependent signals involving CD52-positive immune cells. [1](https://ashpublications.org/blood/article/109/11/4856/23088/T-cell-dependent-survival-of-CD20-and-CD20-plasma)
- Activation and Regulation: Cell-surface CD52 ligation delivers costimulatory signals that aid T-cell activation, while soluble CD52 can bind inhibitory receptors (like Siglec-10) to calm down immune responses. [1](https://pubmed.ncbi.nlm.nih.gov/28283679/)

Clinical Targeting in Lymphoid Organs

- Alemtuzumab: The humanized monoclonal antibody Alemtuzumab (Campath-1H) targets CD52 and induces the profound, near-total depletion of lymphocytes from secondary lymphoid tissues (such as tonsils and lymph nodes) through complement-dependent cytotoxicity (CDC) and antibody-dependent cellular cytotoxicity (ADCC). [1](https://pubmed.ncbi.nlm.nih.gov/15390358/), [2](https://synapse.patsnap.com/article/what-are-cd52-modulators-and-how-do-they-work), [3](https://ashpublications.org/blood/article/109/11/4856/23088/T-cell-dependent-survival-of-CD20-and-CD20-plasma)

- Applications: This depletion mechanism is utilized to treat chronic lymphocytic leukemia (CLL), cutaneous T-cell lymphomas, multiple sclerosis, and to prevent rejection or graft-versus-host disease in organ and cell transplantations. [1](https://www.sciencedirect.com/topics/neuroscience/cd52), [2](https://pubmed.ncbi.nlm.nih.gov/28283679/), [3](https://www.sciencedirect.com/topics/pharmacology-toxicology-and-pharmaceutical-science/cd52-antigen)


![Lymphoid-Organs](../figures/Lymphoid-Organs-Primary-and-Secondary-Lymphoid-Organs.png)

https://www.zoologyverse.com/lymphoid-organs-primary-and-secondary-lymphoid-organs#google_vignette


![lymphoid_tissues](../figures/lymphoid_tissues.jpg)

Papenfuss, T.L., Rebelatto, M.C., Bolon, B. (2019). Pathology of the Lymphoid System. In: Steinbach, T., Patrick, D., Cosenza, M. (eds) Toxicologic Pathology for Non-Pathologists. Humana, New York, NY. https://doi.org/10.1007/978-1-4939-9777-0_10

![schematic of a lymph node](../figures/schematic_lymph_node.png)

19 - Hematopoietic and Lymphoid Tissues  
Michael Linden, Jerrold M. Ward, Sindhu Cherian  
DOI: 10.1016/B978-0-12-381361-9.00019-6  

https://www.sciencedirect.com/science/chapter/edited-volume/abs/pii/B9780123813619000196
